## Multi head Attention 

In [1]:
import torch

# Your Journey starts with one step
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], #x^1
    [0.55, 0.87, 0.66], #x^2
    [0.57, 0.85, 0.64], #x^3
    [0.22, 0.58, 0.33], #x^4
    [0.77, 0.25, 0.10], #x^5
    [0.05, 0.80, 0.55]] #x^6
)

### Casual attention Module

In [2]:
batch = torch.stack((inputs, inputs), dim = 0)
print(batch.shape)
print(batch.shape[0])
d_in = inputs.shape[-1]
d_out = 2

torch.Size([2, 6, 3])
2


In [3]:
import torch.nn as nn

class CasualAttention(nn.Module):
    '''Implements the casual attention mechanism, 
    which ensures that each token can only attend to previous tokens in the sequence.
    
    Args:
        d_in: The dimensionality of the input token embeddings.
        d_out: The dimensionality of the output token embeddings after attention.
        context_length: The maximum length of the input sequence, used to create the attention mask.
        dropout: The dropout rate applied to the attention weights.
        qkv_bias: Whether to include bias terms in the linear layers for query, key, and value projections.
        register_buffer: A method to register a tensor as a buffer in the module, which is not a parameter but should be part of the module's state.
    
    Forward Pass:
        1. Compute the query, key, and value projections from the input token embeddings.
        2. Calculate the attention scores by performing a dot product between the query and key projections.
        3. Apply the attention mask to prevent attending to future tokens.
        4. Normalize the attention scores using softmax to obtain attention weights.
        5. Apply dropout to the attention weights.
        6. Compute the context vector as a weighted sum of the value projections using the attention weights.
        7. Return the context vector, which contains the attended information for each token in
              the sequence. 
    '''
    def __init__(self, d_in, d_out,context_length, dropout, qkv_bias = False):
        super().__init__()
        self. d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", 
            torch.triu(torch.ones(context_length, context_length), 
            diagonal = 1))


    def forward(self,x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attention_scores = queries @ keys.transpose(1,2)
        attention_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        # print("attention_scores", masked)
        attention_weights = torch.softmax(attention_scores/keys.shape[-1]**0.5, dim =-1)
        attention_weights = self.dropout(attention_weights)
        context_vector = attention_weights @ values
        return context_vector

In [4]:
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CasualAttention(d_in, d_out,context_length,0.0)
context_vector = ca(batch)
print("context_vector shape:", context_vector.shape)
print(context_vector)

context_vector shape: torch.Size([2, 6, 2])
tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)


### Multi head attention wrapper
Now we will define a wrapper class that stacks multiple instances of our above implemented casual attention module

In [5]:
class MultiHeadAttentionWrapper(nn.Module):
    '''A wrapper class for implementing multi-head attention by 
    combining multiple instances of the CasualAttention class.
    
    Args:
        d_in: The dimensionality of the input token embeddings.
        d_out: The dimensionality of the output token embeddings after attention.
        context_length: The maximum length of the input sequence, used to create the attention mask.
        dropout: The dropout rate applied to the attention weights.
        num_heads: The number of attention heads.
        qkv_bias: Whether to include bias terms in the linear layers for query, key, and value projections.
    
    Forward Pass:
        1. Initialize multiple instances of the CasualAttention class based on the specified number of heads.
        2. For each head, compute the context vector by passing the input through the corresponding CasualAttention instance.
        3. Concatenate the context vectors from all heads along the last dimension to combine the information from each head.
        4. Return the concatenated context vector, which contains the attended information from all heads for each token in the sequence.
    '''
    def __init__(self, d_in,d_out, context_length, dropout, num_heads, qkv_bias =  False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CasualAttention(d_in, d_out, context_length, dropout, qkv_bias) for _ in range(num_heads)]
        )
    def forward(self,x):
        head_outputs = [head(x) for head in self.heads]
        # print("head_outputs:", [head_output for head_output in head_outputs])
        return torch.cat(head_outputs, dim = -1)

In [6]:
torch.manual_seed(123)
context_length = batch.shape[1]
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads = 2)
context_vector = mha(batch)
print("context_vector shape:", context_vector.shape)
print(context_vector)


context_vector shape: torch.Size([2, 6, 4])
tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)


In [7]:
torch.manual_seed(123)
context_length = batch.shape[1]
d_in, d_out = 3, 1
mha_1 = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads = 2)
context_vector_1 = mha_1(batch)
print("context_vector shape:", context_vector_1.shape)
print(context_vector_1)

context_vector shape: torch.Size([2, 6, 2])
tensor([[[-0.5740,  0.2216],
         [-0.7320,  0.0155],
         [-0.7774, -0.0546],
         [-0.6979, -0.0817],
         [-0.6538, -0.0957],
         [-0.6424, -0.1065]],

        [[-0.5740,  0.2216],
         [-0.7320,  0.0155],
         [-0.7774, -0.0546],
         [-0.6979, -0.0817],
         [-0.6538, -0.0957],
         [-0.6424, -0.1065]]], grad_fn=<CatBackward0>)


### Multi head attention class Implementation

In [8]:
class MultiHeadAttention(nn.Module):

    def __init__(self, d_in,d_out, context_length, dropout, num_heads, qkv_bias = False):
        super().__init__()
        assert(d_out % num_heads == 0), "d_out must be divisible by num_heads"
        

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Dimension of each head

        self. W_query = nn.Linear(d_in, d_out, bias = qkv_bias)
        self. W_key = nn.Linear(d_in, d_out, bias = qkv_bias)
        self. W_value = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out) # Output projection layer
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length,context_length), diagonal = 1)
        )

    def forward(self,x):
        b, num_tokens, d_in = x.shape
        queries = self.W_query(x) # shape ==> [b, num_tokens,d_out]
        keys = self.W_key(x)
        values = self.W_value(x)

        # Reshape for multi-head attention
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) # shape ==> [b, num_tokens, num_heads , head_dim] ... d_out = num_heads * head_dim 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        keys = keys.transpose(1,2) # shape ==> [b, num_heads ,num_tokens, head_dim]
        queries = queries.transpose(1,2)
        values = values.transpose(1,2)

        # calculating attention scores and weights

        attention_scores = queries @ keys.transpose(2,3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attention_scores.masked_fill_(mask_bool, -torch.inf)

        attention_weights = torch.softmax(attention_scores / self.head_dim**0.5, dim = -1)
        attention_weights = self.dropout(attention_weights)
        context_vector = (attention_weights @ values).transpose(1,2) # shape ==> [b, num_tokens, num_heads, head_dim]
        context_vector = context_vector.contiguous().view(b, num_tokens, self.d_out) # shape ==> [b, num_tokens, d_out]

        context_vector = self.out_proj(context_vector)

        return context_vector

In [9]:
torch.manual_seed(123)
batch_size, num_tokens, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, num_tokens, 0.0, num_heads = 2)
context_vector = mha(batch)
print("context_vector shape:", context_vector.shape)
print(context_vector)   

context_vector shape: torch.Size([2, 6, 2])
tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)


#### Reference to above output shape transformations
* keys shape - before reshaping : torch.Size([2, 6, 2])
* keys shape - after reshaping : torch.Size([2, 6, 2, 1])
* keys shape - transposed : torch.Size([2, 2, 6, 1])
* attention_scores shape: torch.Size([2, 2, 6, 6])
* attention_weights shape: torch.Size([2, 2, 6, 6])
* context_vector shape: torch.Size([2, 6, 2, 1])
* context_vector shape after contiguous: torch.Size([2, 6, 2])
* context_vector shape output: torch.Size([2, 6, 2])


In [11]:
# Initializing GPT-2 size attention module 
torch.manual_seed(123)
batch_big = torch.rand(2, 1024 ,768)
b, num_tokens, d_in = batch_big.shape


mha_big = MultiHeadAttention(d_in, d_out, num_tokens, 0.0, num_heads = 12)
context_vector_big = mha_big(batch_big)
print("context_vector_big shape:", context_vector_big.shape)  


context_vector_big shape: torch.Size([2, 1024, 768])
